# Robustness checks: how sure are the scoreboard numbers?

Every number in the notebook 05 comparison came from one 80/20 split. That is a
single run, and two of its claims deserve pressure: that the logistic router really
out-scores the best fixed single model, and that the strategy ordering is not an
accident of which 2,434 prompts landed in the test set. Both checks here resample
the recorded outcomes, so they cost nothing to run and change no model.

The tool is the bootstrap: resample the test prompts with replacement many times,
recompute every strategy's mean cost and mean score on each resample, and look at
the spread. If a gap between two strategies survives across thousands of resamples,
it is a property of the strategies, not of the draw.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

proc = Path("..") / "data" / "processed"
long_df = pd.read_parquet(proc / "records_long.parquet")
core = long_df[~long_df.is_reference]
openrouter = long_df[long_df.is_reference]
model_df = pd.read_parquet(proc / "prompts_features_labels_tau100.parquet")
router_preds = pd.read_parquet(proc / "router_test_predictions.parquet")
reg_picks = pd.read_parquet(proc / "regressor_test_picks.parquet")

# identical split to notebooks 03-05
train_df, test_df = train_test_split(
    model_df, test_size=0.2, random_state=42, stratify=model_df.dataset)
sid = test_df.prompt_id
print(f"{len(sid)} test prompts")

test_rows = core[core.prompt_id.isin(sid)]
lookup = test_rows.set_index(["prompt_id", "model"])[["cost", "score"]]
fixed = test_rows.groupby("model").agg(cost=("cost", "mean"), score=("score", "mean"))

picks = {
    "always cheapest": pd.Series(fixed.cost.idxmin(), index=sid.values),
    "always strongest": pd.Series(fixed.score.idxmax(), index=sid.values),
    "best single value model (fixed)": pd.Series("qwen3-235b-a22b-2507", index=sid.values),
    "logistic router (tuned)": router_preds.set_index("prompt_id").logistic_tuned.loc[sid].rename_axis(None),
    "random forest router (tuned)": router_preds.set_index("prompt_id").random_forest_tuned.loc[sid].rename_axis(None),
    "regressor-derived router": reg_picks.set_index("prompt_id").regressor_routing.loc[sid].rename_axis(None),
    "oracle label": test_df.set_index("prompt_id").label.loc[sid].rename_axis(None),
}

# per-prompt cost and score arrays for each strategy, aligned on the same prompt order
order = list(sid.values)
mat = {}
for name, p in picks.items():
    got = lookup.loc[list(zip(order, p.loc[order].values))]
    mat[name] = (got.cost.to_numpy(), got.score.to_numpy())
or_rows = openrouter[openrouter.prompt_id.isin(sid)].set_index("prompt_id").loc[order]
mat["OpenRouter (reference)"] = (or_rows.cost.to_numpy(), or_rows.score.to_numpy())
print("strategies:", len(mat))


2434 test prompts
strategies: 8


## Bootstrap: 5,000 resamples of the test set

Each resample draws 2,434 prompt indices with replacement and recomputes every
strategy's mean cost and mean score on that draw. Using the same indices for all
strategies matters: the strategies face identical traffic in every resample, the
same condition the original comparison imposed. The seed is fixed so the notebook
reproduces exactly.


In [2]:
rng = np.random.default_rng(42)
B = 5000
n = len(order)
idx = rng.integers(0, n, size=(B, n))

boot = {}
for name, (c, s) in mat.items():
    boot[name] = (c[idx].mean(axis=1), s[idx].mean(axis=1))

rows = []
for name, (bc, bs) in boot.items():
    rows.append({
        "strategy": name,
        "mean cost (USD)": mat[name][0].mean(),
        "cost 95% CI": f"[{np.percentile(bc, 2.5):.4f}, {np.percentile(bc, 97.5):.4f}]",
        "mean score": mat[name][1].mean(),
        "score 95% CI": f"[{np.percentile(bs, 2.5):.3f}, {np.percentile(bs, 97.5):.3f}]",
    })
ci_table = pd.DataFrame(rows).sort_values("mean cost (USD)").set_index("strategy")
ci_table.round(4)


,mean cost (USD),cost 95% CI,mean score,score 95% CI
strategy,,,,
regressor-derived router,0.0006,"[0.0006, 0.0007]",0.5129,"[0.493, 0.532]"
always cheapest,0.0008,"[0.0007, 0.0009]",0.4287,"[0.409, 0.448]"
best single value model (fixed),0.0009,"[0.0008, 0.0009]",0.5378,"[0.518, 0.558]"
oracle label,0.0055,"[0.0044, 0.0067]",0.8215,"[0.806, 0.837]"
random forest router (tuned),0.0073,"[0.0062, 0.0085]",0.5362,"[0.516, 0.556]"
logistic router (tuned),0.0115,"[0.0101, 0.0131]",0.5637,"[0.544, 0.584]"
OpenRouter (reference),0.0225,"[0.0212, 0.0239]",0.4953,"[0.476, 0.515]"
always strongest,0.0615,"[0.0582, 0.0648]",0.5968,"[0.577, 0.616]"


## The claim that matters: logistic router vs the best fixed model

The paper's discussion leans on one comparison: the logistic router's quality edge
over sending everything to qwen3-235b-a22b-2507. A paired bootstrap answers whether
that edge is real: on each resample, compute the score difference between the two
strategies on the same drawn prompts, and count how often it stays positive.


In [3]:
diff = mat["logistic router (tuned)"][1] - mat["best single value model (fixed)"][1]
boot_diff = diff[idx].mean(axis=1)
print(f"observed score difference: {diff.mean():+.4f}")
print(f"95% CI: [{np.percentile(boot_diff, 2.5):+.4f}, {np.percentile(boot_diff, 97.5):+.4f}]")
print(f"share of resamples where the router wins: {(boot_diff > 0).mean():.1%}")

# same check for the router's price premium
dcost = mat["logistic router (tuned)"][0] - mat["best single value model (fixed)"][0]
boot_dcost = dcost[idx].mean(axis=1)
print(f"\nobserved cost difference: {dcost.mean():+.5f} USD per prompt")
print(f"95% CI: [{np.percentile(boot_dcost, 2.5):+.5f}, {np.percentile(boot_dcost, 97.5):+.5f}]")


observed score difference: +0.0259
95% CI: [+0.0168, +0.0353]
share of resamples where the router wins: 100.0%

observed cost difference: +0.01065 USD per prompt
95% CI: [+0.00926, +0.01222]


The paired bootstrap settles the paper's central comparison. The logistic router's
score edge over the fixed qwen model was +0.026, the 95% interval [+0.017, +0.035]
never touches zero, and the router won in 100% of 5,000 resamples. The price
premium is equally real: about a cent more per prompt, interval well clear of zero.
So the honest statement is that the router buys a real quality edge at a real cost,
neither side of that trade is sampling noise.

The intervals also sharpen a verdict notebook 05 could only assert. The random
forest's score interval [0.516, 0.556] sits almost exactly on the fixed model's
[0.518, 0.558]: statistically indistinguishable on quality while costing eight
times more, which is a cleaner way of saying the forest loses outright. And
OpenRouter's interval falls below every trained strategy while costing the most of
the three, so the commercial-reference result survives resampling too.


## Where routing wins and loses: per-category breakdown

The menu-knowledge story predicts routing should earn its keep only where the cheap
default fails. Splitting the same test prompts by source dataset shows exactly
where the logistic router's picks diverge from the fixed model and whether the
divergence pays.


In [4]:
cat = test_df.set_index("prompt_id").dataset.loc[order].to_numpy()
rows = []
for d in sorted(set(cat)):
    m = cat == d
    rows.append({
        "dataset": d,
        "prompts": int(m.sum()),
        "router score": mat["logistic router (tuned)"][1][m].mean(),
        "fixed qwen score": mat["best single value model (fixed)"][1][m].mean(),
        "score edge": mat["logistic router (tuned)"][1][m].mean() - mat["best single value model (fixed)"][1][m].mean(),
        "router cost": mat["logistic router (tuned)"][0][m].mean(),
        "fixed qwen cost": mat["best single value model (fixed)"][0][m].mean(),
        "% routed away from qwen": (picks["logistic router (tuned)"].loc[order].to_numpy()[m] != "qwen3-235b-a22b-2507").mean(),
    })
breakdown = pd.DataFrame(rows).set_index("dataset").sort_values("score edge", ascending=False)
breakdown.round(3)


,prompts,router score,fixed qwen score,score edge,router cost,fixed qwen cost,% routed away from qwen
dataset,,,,,,,
hle,432,0.243,0.095,0.148,0.060,0.002,1.00
aime,12,0.917,0.833,0.083,0.002,0.006,1.00
arenahard,150,0.747,0.747,0.000,0.001,0.001,0.00
gpqa,40,0.600,0.600,0.000,0.000,0.000,0.00
livemathbench,24,0.833,0.833,0.000,0.002,0.002,0.00
mmlupro,600,0.832,0.832,0.000,0.001,0.001,0.00
simpleqa,865,0.519,0.519,0.000,0.000,0.000,0.00
livecodebench,211,0.649,0.654,-0.005,0.005,0.001,1.00
swe-bench,100,0.150,0.160,-0.010,0.002,0.003,0.08


The breakdown turns the menu thesis into a two-line rule. On five of the nine
categories, covering about seven in ten test prompts (simpleqa, mmlupro, arenahard,
gpqa, livemathbench), the router sends 100% of traffic to qwen and matches the
fixed strategy by construction. Its entire quality edge comes from the categories
where it escalates: hle, where routing every prompt to stronger models earns +0.148
on 432 prompts at roughly thirty times the cost, and aime, where the routed choice
is both better and cheaper. The two code categories are the cautionary tale:
diverging from qwen on livecodebench and swe-bench made things slightly worse.

So what the classifier actually learned is: default to the best cheap model, and
escalate the hardest research questions. That a tuned multiclass model converged on
a policy this simple is the per-category version of the paper's conclusion, most of
the value of routing is knowing which one model to default to, and the residual
value is knowing when to leave it.
